# Train baseling config for all tasks on kaggle

In [ ]:
!git clone https://github.com/angelvalen/gpt2-from-scratch.git
%cd gpt2-from-scratch

In [ ]:
!pip install -r requirements.txt

In [ ]:
# Ensure GPU
import torch
assert torch.cuda.is_available()
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
import subprocess

Baseline config for all tasks

# Sentiment

## SST

In [ ]:
BASELINE = {
    "--patience": "5",
    "--fine-tune-mode": "full-model",
    "--hidden_dropout_prob": "0.1",
    "--lr": "1e-5",
    "--batch_size": "32",
    "--weight_decay": "0.0",
    "--model_size": "gpt2-medium", # Showed significant better acc, could sweep small but low on time.
}
FLAGS = ["--use_gpu"]

SWEEPS = {
    "--lr": ["1e-6", "1e-4"],
    "--weight_decay": ["0.01", "0.1"],
    "--hidden_dropout_prob": ["0.0", "0.3"],
}

for param, values in SWEEPS.items():
    for val in values:
        args = {**BASELINE, param: val}
        cmd = ["python", "classifier.py"] + FLAGS
        for k, v in args.items():
            cmd += [k, str(v)]
        subprocess.run(cmd)

## CFIMDB

In [ ]:
BASELINE = {
    "--patience": "5",
    "--fine-tune-mode": "full-model",
    "--hidden_dropout_prob": "0.1",
    "--lr": "1e-5",
    "--batch_size": "8",
    "--weight_decay": "0.0",
    "--model_size": "gpt2", # gpt-med didnt show significant improvement worth the time cost
}
FLAGS = ["--use_gpu"]

SWEEPS = {
    "--lr": ["1e-6", "1e-4"],
    "--weight_decay": ["0.01", "0.1"],
    "--hidden_dropout_prob": ["0.0", "0.3"],
}

for param, values in SWEEPS.items():
    for val in values:
        args = {**BASELINE, param: val}
        cmd = ["python", "classifier.py"] + FLAGS
        for k, v in args.items():
            cmd += [k, str(v)]
        subprocess.run(cmd)

# Paraphrase

IMPORTANT: First train last linear, so that full model .pt persists over it.

In [ ]:
# Last linear layer paraphrase
!python paraphrase_detection.py --use_gpu --small_datasets --paraphrase_dropout_prob 0.3 --lr 1e-3 --batch_size 64

In [ ]:
# Full model paraphrase
!python paraphrase_detection.py --use_gpu --small_datasets --fine-tune-mode full-model --batch_size 32

# Sonnet

In [ ]:
# Top p sampling 
!python sonnet_generation.py --use_gpu --generation_method top_p --batch_size 8

In [ ]:
# Beam search (just gen)
!python sonnet_generation.py --generate_only --use_gpu --generation_method beam

In [ ]:
import shutil

for dir in ["checkpoints", "sonnet_results", "paraphrase_results", "sentiment_results"]:
        
    shutil.make_archive(
        f"/kaggle/working/gpt2-from-scratch/{dir}",
        "zip",
        f"/kaggle/working/gpt2-from-scratch/{dir}"
    )